In [3]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import pickle
from sklearn.model_selection import train_test_split

In [4]:
CSV = r'.\csvfiles'
PKL = r'.\picklefiles'

In [5]:
with open(f"{PKL}\\baselinedataset_trainsmall.pkl", 'rb') as f:
    dataset = pickle.load(f)

In [7]:
with open(f"{PKL}\\baselinedataset_test.pkl", 'rb') as f:
    test_dataset = pickle.load(f)

In [8]:
#Undersampling to meet class imbalances for class 0 and 1
X = dataset[:,:-1]
X_test = test_dataset[:,:-1]
sc = StandardScaler()
X = sc.fit_transform(X)
y = dataset[:,-1]
y_test = test_dataset[:, -1]
#undersample = RandomUnderSampler(sampling_strategy='majority')
#X, y = undersample.fit_resample(X, y)
#X_test, y_test = undersample.fit_resample(X_test, y_test)

In [9]:
count_neg = 0
for el in y:
    if el == 0:
        count_neg += 1

In [10]:
count_pos = y.shape[0] - count_neg

In [11]:
scale_ratio = count_neg / count_pos

In [12]:
#Train test split
#X_train,X_test,y_train,y_test = train_test_split(X, y,test_size = 0.3, random_state = 42)
X_train = X
y_train = y
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'binary:logistic',  
    'eval_metric': 'logloss',        
    'max_depth': 3,                  
    'learning_rate': 0.1,            
    'subsample': 0.8,                
    'colsample_bytree': 0.8,         
    'seed': 42,
    'scale_pos_weight':scale_ratio #class imbalance
} 
num_rounds = 100  
model = xgb.train(params, dtrain, num_rounds)
y_pred = model.predict(dtest)
y_pred_binary = [1 if pred > 0.5 else 0 for pred in y_pred]  # Convert probabilities to binary predictions

print("Accuracy:", accuracy_score(y_test, y_pred_binary))
print("\nClassification Report:\n", classification_report(y_test, y_pred_binary))

Accuracy: 0.5188623733246159

Classification Report:
               precision    recall  f1-score   support

         0.0       0.83      0.53      0.65     25623
         1.0       0.16      0.46      0.24      4967

    accuracy                           0.52     30590
   macro avg       0.50      0.49      0.44     30590
weighted avg       0.72      0.52      0.58     30590



In [14]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred_binary)

array([[13605, 12018],
       [ 2700,  2267]], dtype=int64)

In [15]:
import pandas as pd
test_id = pd.read_csv(f'{CSV}\\Test_ID.csv')
test_id['Predicted Label'] = y_pred_binary
test_id['Predicted Probabilities'] = y_pred

In [16]:
test_id.to_csv(f'{CSV}\\Predictions_trainsmall_XGboost.csv', index = False)

In [17]:
#Implementing Light GBM
import lightgbm as lgb
from sklearn.metrics import  roc_auc_score

# Create a LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'binary',  # for binary classification 
    'metric': 'auc', # area under the curve
    'boosting_type': 'gbdt',    # traditional Gradient Boosting Decision Tree
    'num_leaves': 31,           # number of leaves in one tree
    'learning_rate': 0.05,      # learning rate
    'feature_fraction': 0.9,    # feature fraction
    'bagging_fraction': 0.8,    # bagging fraction
    'bagging_freq': 5,          # bagging frequency
    'verbose': 0,               # 0 for silent mode
    'scale_pos_weight': scale_ratio
}
# Train the model
num_round = 100  # Number of boosting rounds
bst = lgb.train(params, train_data, num_round, valid_sets = [test_data])
# Make predictions on the test set
y_pred_prob_lgb = bst.predict(X_test, num_iteration=bst.best_iteration)
y_pred_lgb = [1 if pred > 0.5 else 0 for pred in y_pred_prob_lgb]  # Convert probabilities to binary predictions

#Model evaluation 
accuracy = accuracy_score(y_test, y_pred_lgb)
roc_auc = roc_auc_score(y_test, y_pred_prob_lgb)

print(f'Accuracy on Test Set: {accuracy:.4f}')
print(f'ROC AUC on Test Set: {roc_auc:.4f}')
print("\nClassification Report:\n", classification_report(y_test, y_pred_lgb))


Accuracy on Test Set: 0.4667
ROC AUC on Test Set: 0.4842

Classification Report:
               precision    recall  f1-score   support

         0.0       0.83      0.46      0.59     25623
         1.0       0.16      0.52      0.24      4967

    accuracy                           0.47     30590
   macro avg       0.49      0.49      0.41     30590
weighted avg       0.72      0.47      0.53     30590



In [18]:
cm = confusion_matrix(y_test, y_pred_lgb)

In [19]:
import pandas as pd
test_id = pd.read_csv(f'{CSV}\\Test_ID.csv')
test_id['Predicted Label'] = y_pred_lgb
test_id['Predicted Probabilities'] = y_pred_prob_lgb

In [20]:
test_id.to_csv(f'{CSV}\\Predictions_trainsmall_LightGBM.csv', index = False)

In [24]:
#Lazy Classifier:
import lazypredict
from lazypredict.Supervised import LazyClassifier

In [25]:
clf = LazyClassifier(verbose=0,ignore_warnings=True, custom_metric=None)
models,predictions = clf.fit(X_train, X_test, y_train, y_test)

  0%|          | 0/29 [00:56<?, ?it/s]


KeyboardInterrupt: 